# P6 — Legal Contract Q&A: Hybrid Search and Re-ranking
Synthetic contract snippets for technical evaluation only; not legal advice.

In [52]:
!pip -q install rank-bm25 sentence-transformers

In [53]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path(
    "/kaggle/input/datasets/babblu123/"
    "real-world-ml-genai-kaggle-complete/"
    "real_world_ml_genai_lab"
)

DATA_DIR = BASE_DIR / "datasets"
DOCS_DIR = BASE_DIR / "docs"

# Check which files are actually available
print("Docs folder exists:", DOCS_DIR.exists())
print("Docs files:", [p.name for p in DOCS_DIR.iterdir()])
print("Dataset files:", [p.name for p in DATA_DIR.iterdir()])

# Load the contract, with a clear error if the filename differs
contract_path = DOCS_DIR / "p6_contracts.txt"

if not contract_path.exists():
    raise FileNotFoundError(
        f"Contract file not found: {contract_path}\n"
        f"Available documents: {[p.name for p in DOCS_DIR.iterdir()]}"
    )

contract_text = contract_path.read_text(encoding="utf-8")
queries = pd.read_csv(DATA_DIR / "p6_legal_queries.csv")

print("Contract characters:", len(contract_text))
print("Legal queries loaded:", len(queries))
display(queries.head())

Docs folder exists: True
Docs files: ['p10_logistics_policy.md', 'p5_clinical_guidelines.md', 'p5_sample_note.txt', 'p6_contracts.txt', 'p7_market_outlook.txt']
Dataset files: ['p10_eval_scenarios.csv', 'p10_shipments.csv', 'p1_iot_telemetry.csv', 'p2_fintech_transactions.csv', 'p3_supply_demand.csv', 'p4_support_tickets.csv', 'p6_legal_queries.csv', 'p7_portfolios.csv', 'p9_claims.jsonl']
Contract characters: 977
Legal queries loaded: 4


,query,expected_clause
0,What is the payment deadline in Contract A?,4.2
1,Which clause covers termination for uncured ma...,9.3
2,How much notice is needed to prevent renewal?,3.5
3,What is the liability cap?,12.4


In [54]:
import re, numpy as np, pandas as pd
chunks=[x.strip() for x in re.split(r"(?=Clause \d+\.\d+)",contract_text) if x.strip()]
from rank_bm25 import BM25Okapi
tokenized=[re.findall(r"\w+",x.lower()) for x in chunks]
bm25=BM25Okapi(tokenized)
from sentence_transformers import SentenceTransformer
embedder=SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
E=embedder.encode(chunks,normalize_embeddings=True)
def hybrid_search(q,k=5):
    bt=bm25.get_scores(re.findall(r"\w+",q.lower()))
    ve=embedder.encode([q],normalize_embeddings=True)[0] @ E.T
    br=np.argsort(bt)[::-1]; vr=np.argsort(ve)[::-1]
    # Reciprocal Rank Fusion
    fused={}
    for rank,idx in enumerate(br): fused[idx]=fused.get(idx,0)+1/(60+rank+1)
    for rank,idx in enumerate(vr): fused[idx]=fused.get(idx,0)+1/(60+rank+1)
    return sorted([(score,chunks[idx]) for idx,score in fused.items()],reverse=True)[:k]
for _,row in queries.iterrows():
    print("\nQ:",row["query"])
    for score,chunk in hybrid_search(row["query"],3): print(round(score,4),chunk)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


Q: What is the payment deadline in Contract A?
0.0325 CONTRACT A — SERVICE AGREEMENT
0.032 Clause 4.2 — Payment Terms: Invoices are payable within thirty (30) calendar days of receipt.
0.0317 Clause 12.4 — Liability Cap: Except for excluded claims, aggregate liability shall not exceed fees paid in the preceding twelve months.

CONTRACT B — SOFTWARE LICENSE

Q: Which clause covers termination for uncured material breach?
0.0328 Clause 9.3 — Termination: Either party may terminate for material breach if the breach remains uncured for 30 days after written notice.
0.032 Clause 6.2 — Data Protection: Customer data must be processed only for documented service purposes.
0.0318 Clause 12.4 — Liability Cap: Except for excluded claims, aggregate liability shall not exceed fees paid in the preceding twelve months.

CONTRACT B — SOFTWARE LICENSE

Q: How much notice is needed to prevent renewal?
0.0328 Clause 3.5 — Renewal: The agreement renews for one year unless either party provides written n

In [55]:
# Optional cross-encoder re-ranking
from sentence_transformers import CrossEncoder
reranker=CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
def rerank(query,candidates):
    pairs=[[query,chunk] for _,chunk in candidates]
    scores=reranker.predict(pairs)
    return sorted(zip(scores,[c for _,c in candidates]),reverse=True)
q=queries.iloc[0]["query"]
print(rerank(q,hybrid_search(q,5))[:3])

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

[(np.float32(-2.7611191), 'Clause 4.2 — Payment Terms: Invoices are payable within thirty (30) calendar days of receipt.'), (np.float32(-3.9970107), 'CONTRACT A — SERVICE AGREEMENT'), (np.float32(-6.9284325), 'Clause 12.4 — Liability Cap: Except for excluded claims, aggregate liability shall not exceed fees paid in the preceding twelve months.\n\nCONTRACT B — SOFTWARE LICENSE')]


## Evaluation
Compare top-3 retrieval with expected clause IDs. Record whether the correct clause appears in retrieved context (context recall proxy) and whether top result is correct (precision@1 proxy). Cite clause text in every answer.